In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [122]:
selected_column = [
    'budget', 'genres', 'release_date', 'revenue', 
    'runtime', 'directors', 'cast', 'popularity'
]
df = pd.read_csv('dataset/TMDB_IMDB_Movies_Dataset.csv',usecols=selected_column)
df.head(n=5)

,release_date,revenue,runtime,budget,popularity,genres,directors,cast
0,2010-07-15,825532764,148,160000000,83.952,"Action, Science Fiction, Adventure",Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,2014-11-05,701729206,169,165000000,140.241,"Adventure, Drama, Science Fiction",Christopher Nolan,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,2008-07-16,1004558444,152,185000000,130.643,"Drama, Action, Crime, Thriller",Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,2009-12-15,2923706026,162,237000000,79.932,"Action, Adventure, Fantasy, Science Fiction",James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,2012-04-25,1518815515,143,220000000,98.082,"Science Fiction, Action, Adventure",Joss Whedon,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [123]:
df = df.dropna(subset=['revenue', 'budget', 'release_date'])
print(f"Movies remaining after removing NaN rows: {len(df)}")

df = df[(df['revenue'] > 10000) & (df['budget'] > 10000)]
print(f"Movies remaining after cleaning missing/placeholder financials: {len(df)}")

Movies remaining after removing NaN rows: 414849
Movies remaining after cleaning missing/placeholder financials: 9554


In [124]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df.dropna(subset=['release_date']) # Drop any rows where the date was completely invalid
df[df['release_date'].isna()].head(n=5)

,release_date,revenue,runtime,budget,popularity,genres,directors,cast


In [125]:
df = df.dropna(subset=['release_date']) # Drop any rows where the date was completely invalid
df.head(n=5)

,release_date,revenue,runtime,budget,popularity,genres,directors,cast
0,2010-07-15,825532764,148,160000000,83.952,"Action, Science Fiction, Adventure",Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,2014-11-05,701729206,169,165000000,140.241,"Adventure, Drama, Science Fiction",Christopher Nolan,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,2008-07-16,1004558444,152,185000000,130.643,"Drama, Action, Crime, Thriller",Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,2009-12-15,2923706026,162,237000000,79.932,"Action, Adventure, Fantasy, Science Fiction",James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,2012-04-25,1518815515,143,220000000,98.082,"Science Fiction, Action, Adventure",Joss Whedon,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [126]:
df = df.sort_values('release_date').reset_index(drop=True)
df.head(n=5)

,release_date,revenue,runtime,budget,popularity,genres,directors,cast
0,1914-04-25,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We..."
1,1914-11-15,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,..."
2,1915-01-04,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter..."
3,1915-02-08,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper..."
4,1915-12-13,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James..."


In [127]:
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month
df.head(n=5)

,release_date,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month
0,1914-04-25,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",1914,4
1,1914-11-15,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",1914,11
2,1915-01-04,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1915,1
3,1915-02-08,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",1915,2
4,1915-12-13,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",1915,12


In [128]:
df = df.drop('release_date', axis=1)
df.head(n=5)

,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month
0,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",1914,4
1,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",1914,11
2,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1915,1
3,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",1915,2
4,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",1915,12


In [129]:
df[df['directors'].isna() | df['cast'].isna()].head(n=10)

,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month
1054,62700,82,157000,0.840,"Adventure, Animation, Fantasy",Payut Ngaokrachang,NaN,1979,4
1481,65993,1,65993,0.600,Animation,NaN,NaN,1985,7
1749,589244,99,2500000,7.115,Documentary,Godfrey Reggio,NaN,1988,4
2284,10000000,94,1000000,1.874,Music,NaN,"Mark Knopfler, John Illsley, Alan Clark, Paul ...",1993,5
2311,15107,3,15107,0.600,"Animation, Comedy",Keith Alcorn,NaN,1993,7
3223,27378,6,27378,0.600,Animation,Stéphane Aubier,NaN,1999,8
3317,21515,1,21515,0.600,"Animation, Comedy",Derek Flood,NaN,2000,2
3623,2074000,101,300000,0.818,Documentary,Pan Nalin,NaN,2001,9
4078,130392,0,100000,0.600,Documentary,Ömer Ali Kazma,NaN,2003,10
4720,178262620,0,45000000,8.497,NaN,"Matthew Asner, Danny Gold",NaN,2006,4


In [130]:
df['primary_director'] = df['directors'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
df['lead_actor'] = df['cast'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
df.head(n=5)

,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month,primary_director,lead_actor
0,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",1914,4,Herbert Brenon,Annette Kellerman
1,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",1914,11,Cecil B. DeMille,Bessie Barriscale
2,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1915,1,Cecil B. DeMille,Mabel Van Buren
3,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",1915,2,D.W. Griffith,Henry B. Walthall
4,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",1915,12,Cecil B. DeMille,Fannie Ward


In [131]:
df['director_hist_rev'] = df.groupby('primary_director')['revenue'].transform(
    lambda x: x.expanding().mean().shift()
)
df['actor_hist_rev'] = df.groupby('lead_actor')['revenue'].transform(
    lambda x: x.expanding().mean().shift()
)
df.head(n=5)

,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month,primary_director,lead_actor,director_hist_rev,actor_hist_rev
0,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",1914,4,Herbert Brenon,Annette Kellerman,NaN,NaN
1,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",1914,11,Cecil B. DeMille,Bessie Barriscale,NaN,NaN
2,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1915,1,Cecil B. DeMille,Mabel Van Buren,87028.0,NaN
3,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",1915,2,D.W. Griffith,Henry B. Walthall,NaN,NaN
4,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",1915,12,Cecil B. DeMille,Fannie Ward,94626.0,NaN


In [132]:
df.loc[df['primary_director'] == 'Unknown', 'director_hist_rev'].head(500)

1481             NaN
2284    6.599300e+04
4979    5.032996e+06
5572    3.658664e+06
5605    3.816503e+06
5709    3.413203e+06
5747    3.699084e+06
6026    3.444929e+06
8398    3.139313e+06
8857    3.079389e+06
9263    3.327118e+06
9264    3.026107e+06
9265    2.775265e+06
9268    2.563014e+06
9396    2.381084e+06
9398    2.229012e+06
9551    2.095949e+06
9552    5.912919e+07
9553    1.098254e+08
Name: director_hist_rev, dtype: float64

In [133]:
df.loc[df['lead_actor'] == 'Unknown', 'actor_hist_rev'].head(500)

1054             NaN
1481    6.270000e+04
1749    6.434650e+04
2311    2.393123e+05
3223    1.832610e+05
3317    1.520844e+05
3623    1.303228e+05
4078    4.079910e+05
4720    3.732911e+05
4749    2.013877e+07
4979    1.814443e+07
4998    1.657767e+07
5091    1.524773e+07
6596    1.407633e+07
6803    1.312502e+07
6897    2.691669e+07
7265    2.575444e+07
7857    8.061594e+07
7944    7.619283e+07
7964    1.248143e+08
7972    1.185752e+08
8506    1.129302e+08
8661    1.077981e+08
9173    1.031181e+08
9190    9.886314e+07
9268    9.502861e+07
9363    9.137428e+07
9398    8.873079e+07
9552    8.556541e+07
Name: actor_hist_rev, dtype: float64

In [134]:
df.loc[df['primary_director'] == 'Unknown', 'director_hist_rev'] = np.nan
df.loc[df['lead_actor'] == 'Unknown', 'actor_hist_rev'] = np.nan

In [135]:
df.loc[df['primary_director'] == 'Unknown', 'director_hist_rev'].head(500)

1481   NaN
2284   NaN
4979   NaN
5572   NaN
5605   NaN
5709   NaN
5747   NaN
6026   NaN
8398   NaN
8857   NaN
9263   NaN
9264   NaN
9265   NaN
9268   NaN
9396   NaN
9398   NaN
9551   NaN
9552   NaN
9553   NaN
Name: director_hist_rev, dtype: float64

In [136]:
df.loc[df['lead_actor'] == 'Unknown', 'actor_hist_rev'].head(500)

1054   NaN
1481   NaN
1749   NaN
2311   NaN
3223   NaN
3317   NaN
3623   NaN
4078   NaN
4720   NaN
4749   NaN
4979   NaN
4998   NaN
5091   NaN
6596   NaN
6803   NaN
6897   NaN
7265   NaN
7857   NaN
7944   NaN
7964   NaN
7972   NaN
8506   NaN
8661   NaN
9173   NaN
9190   NaN
9268   NaN
9363   NaN
9398   NaN
9552   NaN
Name: actor_hist_rev, dtype: float64

In [137]:
global_median_rev = df['revenue'].median()

In [138]:
df['is_debut_director'] = df['director_hist_rev'].isna().astype(int)
df['is_debut_actor'] = df['actor_hist_rev'].isna().astype(int)
df.head(n=10)

,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month,primary_director,lead_actor,director_hist_rev,actor_hist_rev,is_debut_director,is_debut_actor
0,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",1914,4,Herbert Brenon,Annette Kellerman,NaN,NaN,1,1
1,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",1914,11,Cecil B. DeMille,Bessie Barriscale,NaN,NaN,1,1
2,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1915,1,Cecil B. DeMille,Mabel Van Buren,8.702800e+04,NaN,0,1
3,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",1915,2,D.W. Griffith,Henry B. Walthall,NaN,NaN,1,1
4,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",1915,12,Cecil B. DeMille,Fannie Ward,9.462600e+04,NaN,0,1
5,102437,60,22472,0.600,"Drama, Romance",Cecil B. DeMille,"Geraldine Farrar, Theodore Roberts, Pedro de C...",1915,12,Cecil B. DeMille,Geraldine Farrar,1.088723e+05,NaN,0,1
6,102768,50,18575,0.640,Drama,Cecil B. DeMille,"Geraldine Farrar, Wallace Reid, Pedro de Cordo...",1916,5,Cecil B. DeMille,Geraldine Farrar,1.072635e+05,102437.0,0,0
7,1750000,197,385907,10.040,"Drama, History",D.W. Griffith,"Lillian Gish, Mae Marsh, Robert Harron, F.A. T...",1916,9,D.W. Griffith,Lillian Gish,1.100000e+07,NaN,0,1
8,1390000,180,1000000,0.887,"Fantasy, Drama",Herbert Brenon,"Annette Kellerman, William E. Shay, William E....",1916,10,Herbert Brenon,Annette Kellerman,5.773000e+06,5773000.0,0,0
9,8000000,85,200000,3.474,"Adventure, Drama, Action, Science Fiction",Stuart Paton,"Jane Gail, Howard Crampton, Matt Moore, Willia...",1916,12,Stuart Paton,Jane Gail,NaN,NaN,1,1


In [139]:
df['director_hist_rev'] = df['director_hist_rev'].fillna(global_median_rev)
df['actor_hist_rev'] = df['actor_hist_rev'].fillna(global_median_rev)
df.head(n=5)

,revenue,runtime,budget,popularity,genres,directors,cast,release_year,release_month,primary_director,lead_actor,director_hist_rev,actor_hist_rev,is_debut_director,is_debut_actor
0,5773000,95,2221000,0.896,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",1914,4,Herbert Brenon,Annette Kellerman,14974848.5,14974848.5,1,1
1,87028,50,16988,0.900,"Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",1914,11,Cecil B. DeMille,Bessie Barriscale,14974848.5,14974848.5,1,1
2,102224,45,15110,3.238,"Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1915,1,Cecil B. DeMille,Mabel Van Buren,87028.0,14974848.5,0,1
3,11000000,193,100000,13.791,"Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",1915,2,D.W. Griffith,Henry B. Walthall,14974848.5,14974848.5,1,1
4,137365,59,17311,3.104,Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",1915,12,Cecil B. DeMille,Fannie Ward,94626.0,14974848.5,0,1


In [140]:
df = df.drop(columns=['directors', 'cast', 'primary_director', 'lead_actor'])
df.head(n=5)

,revenue,runtime,budget,popularity,genres,release_year,release_month,director_hist_rev,actor_hist_rev,is_debut_director,is_debut_actor
0,5773000,95,2221000,0.896,Fantasy,1914,4,14974848.5,14974848.5,1,1
1,87028,50,16988,0.900,"Romance, Western, Adventure",1914,11,14974848.5,14974848.5,1,1
2,102224,45,15110,3.238,"Western, Romance",1915,1,87028.0,14974848.5,0,1
3,11000000,193,100000,13.791,"Drama, History, War",1915,2,14974848.5,14974848.5,1,1
4,137365,59,17311,3.104,Drama,1915,12,94626.0,14974848.5,0,1


In [141]:
df['genres'].astype(str).str.get_dummies(sep=',')
genres_dummies = df['genres'].astype(str).str.get_dummies(sep=', ')
print(len(genres_dummies))
genres_dummies.head(n=5)

9554


,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western,nan
0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0
3,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [142]:
top_genres = genres_dummies.sum().sort_values(ascending=False).head(12).index
print(top_genres)

Index(['Drama', 'Comedy', 'Action', 'Thriller', 'Romance', 'Adventure',
       'Crime', 'Horror', 'Family', 'Science Fiction', 'Fantasy', 'Mystery'],
      dtype='object')


In [143]:
genres_dummies[top_genres]

,Drama,Comedy,Action,Thriller,Romance,Adventure,Crime,Horror,Family,Science Fiction,Fantasy,Mystery
0,0,0,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,1,1,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9549,0,0,0,1,0,0,1,1,0,0,0,0
9550,0,0,0,0,0,0,0,1,0,0,0,0
9551,1,0,1,0,0,0,0,0,0,0,0,0
9552,1,0,1,0,0,0,0,0,0,0,0,0


In [144]:
genres_dummies = genres_dummies[top_genres]
genres_dummies = genres_dummies.add_prefix('genre_')
genres_dummies.head(n=5)

,genre_Drama,genre_Comedy,genre_Action,genre_Thriller,genre_Romance,genre_Adventure,genre_Crime,genre_Horror,genre_Family,genre_Science Fiction,genre_Fantasy,genre_Mystery
0,0,0,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,1,1,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0


In [145]:
df = pd.concat([df, genres_dummies], axis=1)
df = df.drop('genres', axis=1)
df.head(n=5)

,revenue,runtime,budget,popularity,release_year,release_month,director_hist_rev,actor_hist_rev,is_debut_director,is_debut_actor,...,genre_Action,genre_Thriller,genre_Romance,genre_Adventure,genre_Crime,genre_Horror,genre_Family,genre_Science Fiction,genre_Fantasy,genre_Mystery
0,5773000,95,2221000,0.896,1914,4,14974848.5,14974848.5,1,1,...,0,0,0,0,0,0,0,0,1,0
1,87028,50,16988,0.900,1914,11,14974848.5,14974848.5,1,1,...,0,0,1,1,0,0,0,0,0,0
2,102224,45,15110,3.238,1915,1,87028.0,14974848.5,0,1,...,0,0,1,0,0,0,0,0,0,0
3,11000000,193,100000,13.791,1915,2,14974848.5,14974848.5,1,1,...,0,0,0,0,0,0,0,0,0,0
4,137365,59,17311,3.104,1915,12,94626.0,14974848.5,0,1,...,0,0,0,0,0,0,0,0,0,0


In [149]:
df.head(n=10)

,revenue,runtime,budget,popularity,release_year,release_month,director_hist_rev,actor_hist_rev,is_debut_director,is_debut_actor,...,genre_Action,genre_Thriller,genre_Romance,genre_Adventure,genre_Crime,genre_Horror,genre_Family,genre_Science Fiction,genre_Fantasy,genre_Mystery
0,5773000,95,2221000,0.896,1914,4,1.497485e+07,14974848.5,1,1,...,0,0,0,0,0,0,0,0,1,0
1,87028,50,16988,0.900,1914,11,1.497485e+07,14974848.5,1,1,...,0,0,1,1,0,0,0,0,0,0
2,102224,45,15110,3.238,1915,1,8.702800e+04,14974848.5,0,1,...,0,0,1,0,0,0,0,0,0,0
3,11000000,193,100000,13.791,1915,2,1.497485e+07,14974848.5,1,1,...,0,0,0,0,0,0,0,0,0,0
4,137365,59,17311,3.104,1915,12,9.462600e+04,14974848.5,0,1,...,0,0,0,0,0,0,0,0,0,0
5,102437,60,22472,0.600,1915,12,1.088723e+05,14974848.5,0,1,...,0,0,1,0,0,0,0,0,0,0
6,102768,50,18575,0.640,1916,5,1.072635e+05,102437.0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1750000,197,385907,10.040,1916,9,1.100000e+07,14974848.5,0,1,...,0,0,0,0,0,0,0,0,0,0
8,1390000,180,1000000,0.887,1916,10,5.773000e+06,5773000.0,0,0,...,0,0,0,0,0,0,0,0,1,0
9,8000000,85,200000,3.474,1916,12,1.497485e+07,14974848.5,1,1,...,1,0,0,1,0,0,0,1,0,0


In [167]:
df = df.apply(pd.to_numeric, errors='coerce').dropna()

In [168]:
X = df.drop('revenue', axis=1)
y = df['revenue']

In [169]:
# since df got datetime sort anyway
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training on {len(X_train)} historical movies...")
print(f"Testing on {len(X_test)} newer movies...")

Training on 7643 historical movies...
Testing on 1911 newer movies...


In [171]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    shuffle=False
)
print(f"Training on {len(X_train)} historical movies...")
print(f"Testing on {len(X_test)} newer movies...")

Training on 7643 historical movies...
Testing on 1911 newer movies...


In [211]:
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'
)

model.fit(X_train, y_train)

print("\n--- 6. Evaluating Performance ---")
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Test Set R-Squared: {r2:.2f} (1.0 is a perfect oracle)")
print(f"Mean Absolute Error: ${mae:,.2f} off per prediction")

print("\nTop 5 Most Important Features driving Box Office Revenue:")
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(importance.head(5).to_string(index=False))

# SAVE THE MODEL FOR THE DASHBOARD
model.save_model("xgboost_box_office_model.json")
print("\nSUCCESS: Model trained and saved as 'xgboost_box_office_model.json'")


--- 6. Evaluating Performance ---
Test Set R-Squared: 0.56 (1.0 is a perfect oracle)
Mean Absolute Error: $71,009,304.00 off per prediction

Top 5 Most Important Features driving Box Office Revenue:
        Feature  Importance
         budget    0.399390
     popularity    0.164626
 is_debut_actor    0.070747
genre_Adventure    0.063777
        runtime    0.053912

SUCCESS: Model trained and saved as 'xgboost_box_office_model.json'
